# One-call run

The other guides call the steps one by one: `deposit`, `slow_down`, `beam_torque`, `neutron_rates`. `run_nbi(equilibrium, plasma, beams, ...)` chains them on one set of flux shells and returns **one** `xarray.Dataset` over $\rho$:

- deposition: birth and power density, and the injected, absorbed and shine-through power of each beam (and of each energy component of a `BeamSource`);
- slowing down: electron and ion heating, fast-ion density and pressure, beam-driven current;
- torque: collisional on the electrons and the ions and, in an axisymmetric equilibrium, $\mathbf{j}\times\mathbf{B}$ (on the shells of a 3D equilibrium the collisional part only);
- neutrons: beam-target and thermonuclear emissivity of every neutron-producing reaction of the beam isotope with the plasma's ions (none for a hydrogen beam, then the step is skipped);
- the totals as 0-d variables, and in the attributes what the run was made of.

Each step is the function named above with its own defaults, so the numbers are those of calling them one by one; the physics and the assumptions (zero orbit width, no losses, no charge exchange of the fast ions) are in their guides.

Here: an ITER-like case, a Solov'ev equilibrium of 6.2 m and 5.3 T, a D-T plasma at $n_e = 10^{20}$ m⁻³ and $T_e = T_i = 20$ keV in the centre, and two 1 MeV deuterium beams of 16.5 MW each, tangent to $R = 5.3$ m, one in the midplane and one 0.6 m below it.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge.run import main, run_nbi, summary

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
plasma = foc.Plasma.parabolic(
    1e20, 2e4, ions={"D": (2, 1, 0.48), "T": (3, 1, 0.48), "He": (4, 2, 0.02)}
)
beams = {}
for i, (name, height) in enumerate((("on-axis", 0.0), ("off-axis", -0.6))):
    origin, direction = foc.tangential_injection(5.3, 12.0, height=height)
    beams[name] = foc.NeutralBeam.gaussian(
        origin,
        direction,
        1.0e6,  # eV
        16.5e6,  # W
        mass_number=2,
        width=0.3,
        divergence=0.005,
        markers=1000,
        seed=503 + i,  # as the parameter file below draws them
    )

ds = run_nbi(
    equilibrium,
    plasma,
    beams,
    rho_edges=np.linspace(0.0, 1.0, 21),
    step=0.03,
    births_per_marker=2,
    seed=3,
)
print(summary(ds))

The beams cross the whole plasma at 1 MeV, yet less than 0.1 % of the power shines through. About two thirds of the absorbed power heats the electrons. The beams drive 1.2 MA, and the torques add up to the births' rate of angular momentum but for the small $\mathbf{j}\times\mathbf{B}$ part outside the last closed surface. At 20 keV the D-T plasma makes far more neutrons by itself than the beams do with it.

The dataset holds all of it, each variable with its units and long name:

In [ ]:
ds

## Heating

The deposited power density and where it goes once the fast ions slow down. Near the axis the ions get almost as much as the electrons; further out they get less and less. The critical energy, above which a fast ion heats mainly the electrons, is proportional to $T_e$: in the cooler outer shells more of the slowing down lies above it.

In [ ]:
names = {
    "power_density": "deposited",
    "electron_heating": "electrons",
    "ion_heating": "ions",
}
heating = ds[list(names)].rename(names).to_array("part", name="power density")
heating.attrs["units"] = "W/m^3"
heating.plasma.plot.profiles(x="rho", over="part")

## Fast ions and current

The fast-ion pressure follows the births and the slowing-down time, which grows with $T_e^{3/2}$: it is peaked on the axis. The driven current density, the fast ions' current shielded by the electrons, peaks off the axis and falls to zero at the edge.

In [ ]:
ds.fast_ion_pressure.plasma.plot.lineout(x="rho")

In [ ]:
ds.current_density.plasma.plot.lineout(x="rho")

## Torque

The collisional torque on the electrons and on the ions and the $\mathbf{j}\times\mathbf{B}$ torque of the guiding-centre jump at birth ([Beam torque](/freeofcharge/examples/nbi-torque/)):

In [ ]:
names = {
    "torque_collisional_electrons": "collisional, electrons",
    "torque_collisional_ions": "collisional, ions",
    "torque_jxb": "$j \\times B$",
    "torque": "total",
}
torque = ds[list(names)].rename(names).to_array("component", name="torque density")
torque.attrs["units"] = "N m/m^3"
torque.plasma.plot.profiles(x="rho", over="component")

## Neutrons

A deuterium beam in a D-T plasma reacts with both: T(d,n)⁴He and D(d,n)³He. The emissivity of each, beam-target and thermonuclear, over `reaction`; `neutron_emissivity` is their sum. D-T dominates, its thermonuclear rate by more than two orders of magnitude and its beam-target rate by about one; the thermonuclear rate dominates except in the outermost shells, where $T_i$ is low and the beam-target rate takes over.

In [ ]:
kinds = xr.concat(
    [ds.beam_target_emissivity, ds.thermonuclear_emissivity],
    dim=xr.DataArray(["beam-target", "thermonuclear"], dims="kind", name="kind"),
)
curves = kinds.stack(curve=("reaction", "kind"))
labels = [f"{r}, {k}" for r, k in curves.curve.values]
curves = curves.drop_vars(["curve", "reaction", "kind"]).assign_coords(curve=labels)
curves.name = "neutron emissivity"
curves.attrs = {"long_name": "neutron emissivity", "units": "1/(m^3 s)"}
curves.plasma.plot.profiles(x="rho", over="curve", logy=True)

## Per beam

The deposition of each beam is kept over `source`:

In [ ]:
per_beam = ds[["source_absorbed_power", "source_shine_through_fraction"]].to_dataframe()
per_beam["source_absorbed_power"] /= 1e6
per_beam.rename(
    columns={
        "source_absorbed_power": "absorbed [MW]",
        "source_shine_through_fraction": "shine-through",
    }
)

## Parameter file and command line

The same run as a TOML file: the equilibrium (`kind` = `"solovev"`, `"geqdsk"` with a `path`, or for 3D `"vmec"` with a wout `path` or a ConStellaration `boundary`, `"desc"`, each with an optional `grid` for a `GriddedEquilibrium`), the profiles (`"parabolic"` or `"tabulated"`), the beams (`"tangential"`, `"gaussian"`, `"source"`, `"box"` of sources, or `"markers"` from a beamline's netCDF file), the numerics and the options of each step. All numbers are in SI units, eV and rad.

In [ ]:
folder = Path(tempfile.mkdtemp())
params = folder / "iter_like.toml"
_ = params.write_text(
    """
title = "ITER-like, two 1 MeV D beams"

[equilibrium]
kind = "solovev"
major_radius = 6.2
minor_radius = 1.8
elongation = 1.7
toroidal_field = 5.3
wall_psi_n = 1.3

[plasma]
kind = "parabolic"
ne0 = 1.0e20
te0 = 2.0e4
ions.D = { mass_number = 2, charge = 1, fraction = 0.48 }
ions.T = { mass_number = 3, charge = 1, fraction = 0.48 }
ions.He = { mass_number = 4, charge = 2, fraction = 0.02 }

[[beams]]
kind = "tangential"
name = "on-axis"
energy = 1.0e6
power = 16.5e6
mass_number = 2
tangency_radius = 5.3
start_radius = 12.0
width = 0.3
divergence = 0.005
markers = 1000

[[beams]]
kind = "tangential"
name = "off-axis"
energy = 1.0e6
power = 16.5e6
mass_number = 2
tangency_radius = 5.3
start_radius = 12.0
height = -0.6
width = 0.3
divergence = 0.005
markers = 1000

[numerics]
rho_edges = 20      # equal shells, or a list of edges
step = 0.03         # m
births_per_marker = 2
seed = 3            # beam i's markers: seed + 500 + i

[neutrons]
enabled = "auto"    # true, false, or the reactions that apply
"""
)

`freeofcharge run iter_like.toml -o out.nc` runs it (the console script calls `freeofcharge.run.main`; from Python, as here, call `main` with the arguments). Typos and missing or misplaced keys stop it before anything runs, with the section and key in the message.

In [ ]:
out = folder / "out.nc"
main(["run", str(params), "-o", str(out)])

The file reproduces the Python call above; `freeofcharge summary out.nc` prints the totals of a written file again.

In [ ]:
back = xr.load_dataset(out)
for name in ("absorbed_power", "driven_current", "torque_total", "neutron_rate"):
    assert float(back[name]) == float(ds[name]), name
print(back.attrs["parameter_file"])
main(["summary", str(out)])

A mistake in the file:

In [ ]:
bad = folder / "bad.toml"
_ = bad.write_text(params.read_text().replace("energy = 1.0e6", "energy = '1 MeV'", 1))
main(["run", str(bad)])

On a 3D equilibrium (e.g. `kind = "vmec"` with `grid = [48, 32, 48]`) the same call slows the ions down with the local pitch (no orbit average), leaves the driven current undefined (NaN, with a warning: the trapped-electron correction is a tokamak result) and computes the collisional torque only (`ds.attrs["torque"]`).